# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PHẦN 2: KỸ THUẬT ĐẶC TRƯNG (FEATURE ENGINEERING) & THIẾT LẬP DÂY CHUYỀN XỬ LÍ (PIPELINE)

> **Mục tiêu chính:** 
> - Tạo thêm các thước đo đặc trưng từ nội dung email (viết hoa, dấu cảm thán, tiền tệ, tiền tố trả lời).
> - Chia dữ liệu làm 2 phần: 80% (Train) và 20% (Test).
> - Đóng gói toàn bộ các bước biến đổi dữ liệu thành một dây chuyền tự động (`Pipeline`).


---
## Khai báo Thư viện & Nạp Dữ liệu Sạch từ Notebook 1


In [ ]:
# Khai báo thư viện & Nạp dữ liệu sạch từ Notebook 1
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    pass

# Đảm bảo import từ thư mục gốc dự án
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from scripts.feature_engineering import (
    load_clean_data, extract_domain_features, split_stratified_train_test,
    build_column_transformer, save_pipeline_and_splits, export_train_test_csv
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Nạp dữ liệu sạch qua hàm chuẩn hóa load_clean_data()
df = load_clean_data()
print(f"Nạp thành công: {df.shape[0]:,} dòng, {df.shape[1]} cột")

---
## Mục 4: Kỹ thuật Trích xuất Đặc trưng Miền & Biến đổi (Domain Feature Engineering)

### 4.1 Trích xuất 7 Đặc trưng Miền Nghiệp vụ (Domain Feature Extraction)
**Mục tiêu Kỹ thuật:**
- Dựa trên phát hiện từ EDA (Notebook 1), trích xuất 7 đặc trưng số bổ trợ cho bộ từ vựng TF-IDF:
  1. `text_len`: Chiều dài ký tự văn bản sạch `text_clean`.
  2. `log_text_len`: Biến đổi logarit $\ln(1 + x)$ nén phân phối lệch phải cực đoan của độ dài văn bản.
  3. `caps_ratio`: Tỷ lệ chữ IN HOA trên toàn bộ email (thư rác thường viết hoa nhiều để gây chú ý).
  4. `exclamation_count`: Tần suất xuất hiện dấu chấm than (`!`) đo lường mức độ giật gân, cấp bách.
  5. `dollar_count`: Tần suất ký hiệu tài chính (`$`) thể hiện lời mời chào tiền thưởng hoặc giao dịch đáng ngờ.
  6. `has_re_prefix`: Email có tiền tố trả lời `Re:` ở đầu tiêu đề hay không (1: Có, 0: Không).
  7. `subject_exclamation`: Tiêu đề có chứa dấu `!` hay không (1: Có, 0: Không).


In [ ]:
# 1. Trích xuất 7 đặc trưng miền qua hàm chuẩn hóa từ scripts.feature_engineering
df = extract_domain_features(df)

# Hiển thị mẫu dữ liệu 7 đặc trưng miền trích xuất
feature_preview = ['log_text_len', 'caps_ratio', 'exclamation_count', 'dollar_count', 'has_re_prefix', 'subject_exclamation', 'label']
print("Mẫu 5 dòng đặc trưng miền trích xuất:")
display(df[feature_preview].head())


**Đúc kết trích xuất đặc trưng miền:**
* **Số hóa thành công hành vi:** Cả 7 đặc trưng số học đã được tạo lập đầy đủ và chuẩn hóa tên cột.
* **Đa dạng hóa góc nhìn:** Bổ sung góc nhìn về cấu trúc, độ khẩn cấp và ngữ cảnh đối thoại bên cạnh các đặc trưng tần suất từ ngữ thuần túy của TF-IDF.


---
### 4.2 Biến đổi Nén Phân phối Lệch & Phân nhóm Tứ phân vị (Log Transform & Quantile Binning)
**Mục tiêu Kỹ thuật:**
- Tạo cột độ dài tiêu đề `subject_len` phục vụ kiểm tra đối soát chi tiết.
- Sử dụng hàm logarit tự nhiên $\ln(1 + \text{text\_len})$ để nén phân phối độ dài có đuôi dài cực đoan.
- Rời rạc hóa (`pd.qcut`) độ dài văn bản thành 4 khoảng tứ phân vị bằng nhau (`Ngắn`, `Trung bình`, `Dài`, `Rất dài`) để kiểm tra phân bổ số lượng email.


In [ ]:
# 2. Tạo cột độ dài tiêu đề và chia 4 khoảng tứ phân vị cho độ dài văn bản
df['subject_len'] = df['subject'].fillna('').astype(str).str.len()
df['length_bin'] = pd.qcut(df['text_len'], q=4, labels=['Ngắn', 'Trung bình', 'Dài', 'Rất dài'])

print("Phân bố số lượng email theo 4 nhóm độ dài văn bản:")
display(df['length_bin'].value_counts().rename("Số lượng email"))


**Đúc kết biến đổi logarit & phân nhóm:**
* **Thu hẹp khoảng cách ngoại lai:** Hàm logarit kéo độ dài từ hàng chục nghìn ký tự về dải giá trị $[0, 14]$, giúp bước chuẩn hóa `MinMaxScaler` phía sau không bị các thư siêu dài ép bẹp giá trị.
* **Phân bổ đồng đều:** 4 khoảng tứ phân vị phân chia mẫu cân bằng (khoảng 8.165 email mỗi nhóm), sẵn sàng cho các phân tích đối sánh hành vi.


---
### 4.3 Kiểm chứng Ý nghĩa Phân phối Thống kê Giữa Thư Thường (Ham) & Thư Rác (Spam)
**Mục tiêu Kỹ thuật:**
- Tính toán các chỉ số thống kê mô tả (Mean, Median, Độ lệch chuẩn Std) của 7 đặc trưng miền phân tách theo nhãn `label` (0: Ham, 1: Spam).
- Đối soát giá trị trung bình để xác nhận tính phân biệt trước khi đưa toàn bộ vào Pipeline xử lý.


In [ ]:
# 3. Bảng thống kê phân phối (mean, median, std) theo nhãn Ham (0) vs Spam (1)
domain_stats_cols = ['caps_ratio', 'exclamation_count', 'dollar_count', 'text_len', 'log_text_len', 'has_re_prefix', 'subject_exclamation']
domain_distribution = df.groupby('label')[domain_stats_cols].agg(['mean', 'median', 'std']).round(4).T

print("Bảng thống kê phân phối đặc trưng miền theo nhãn (Ham vs Spam):")
display(domain_distribution)


**Đúc kết kiểm chứng ý nghĩa thống kê:**
* **Tính phân biệt vượt trội của tiền tố tiêu đề:** Đặc trưng `has_re_prefix` xuất hiện ở 55.26% thư Ham nhưng chỉ có 0.75% ở thư Spam (gấp gần 74 lần), phản ánh luồng hội thoại công việc thực tế.
* **Tín hiệu cảnh báo giật gân:** Thư Spam có tỷ lệ xuất hiện dấu chấm than trên tiêu đề cao gấp 5 lần so với Ham (11.06% so với 2.17%).
* **Chữ in hoa và ký hiệu tiền:** Thư Spam có tỷ lệ chữ in hoa và số lượng dấu `$` trung bình vượt trội so với thư Ham.


---
## Mục 5: Phân chia Dữ liệu & Lắp ráp dây chuyền xử lý (Pipeline)

> Kết hợp chuỗi văn bản sạch `text_clean` cùng 7 đặc trưng số học vào một khối tiền xử lý tự động chung (`ColumnTransformer`) để đưa vào mô hình học máy


### 5.1 Phân chia Train & Test
**Mục tiêu Kỹ thuật:**
- Chia dữ liệu theo tỷ lệ giữa tập Train (80%) và tập Test (20%).
- Áp dụng phân tầng theo nhãn (`stratify=y`) với `RANDOM_SEED = 42` nhằm bảo toàn tuyệt đối tỷ lệ phân bố giữa hai lớp, ngăn ngừa rò rỉ dữ liệu (Data Leakage) sang tập kiểm thử.
- Bộ đặc trưng đầu vào gồm: `text_clean` và 7 biến (bao gồm log_text_len): `['caps_ratio', 'exclamation_count', 'dollar_count', 'text_len', 'log_text_len', 'has_re_prefix', 'subject_exclamation']`.


In [ ]:
# Phân chia Train/Test 80/20 phân tầng chống rò rỉ dữ liệu qua hàm chuẩn hóa
X_train, X_test, y_train, y_test = split_stratified_train_test(df, test_size=0.20, random_state=RANDOM_SEED)
print(f"Train: {len(X_train):,} mẫu (Tỷ lệ Spam: {y_train.mean()*100:.2f}%)")
print(f"Test:  {len(X_test):,} mẫu (Tỷ lệ Spam: {y_test.mean()*100:.2f}%)")

**Kết quả chia tập dữ liệu:**
- Đã chia tách độc lập: 26.128 mẫu tập Train và 6.532 mẫu tập Test.
- Tỷ lệ thư rác ở cả hai tập được bảo toàn chuẩn xác ở mức 47,66%.
- Mọi thao tác học từ vựng (TF-IDF), chấm điểm (Chi-Square) và co giãn số học (MinMaxScaler) chỉ được học trên tập Train rồi mới chuyển đổi sang tập Test.


### 5.2 Xây dựng dây chuyền xử lý (Pipline) tự động (`ColumnTransformer`)
Dây chuyền gồm 2 nhánh xử lý chuyên biệt chạy song song:
1. **Nhánh văn bản (`text_clean`):** Dùng `TfidfVectorizer` trích xuất từ đơn và cụm 2 từ (`ngram_range=(1, 2)`), sau đó dùng thuật toán `SelectKBest(chi2)` lọc lấy đúng 2.000 từ có khả năng phân biệt thư rác tốt nhất.
2. **Nhánh số học (`num_cols`):** Dùng `MinMaxScaler` đưa 7 số đo về chung thang điểm $[0, 1]$ để các số lớn không lấn át số nhỏ và tránh giá trị âm gây lỗi cho Naive Bayes.


In [ ]:
# Xây dựng Pipeline Tiền xử lý Đa tầng qua hàm chuẩn hóa build_column_transformer()
preprocessor = build_column_transformer(max_features=5000, k_best=2000, ngram_range=(1, 2))


**Kết quả thiết lập Pipeline:**
- Pipeline đa tầng kết hợp thành công nhánh văn bản cùng nhánh số học tạo ra không gian biểu diễn gồm đúng 2.007 đặc trưng (2.000 từ vựng quan trọng + 7 đặc trưng số).
- Khối `ColumnTransformer` đóng vai trò kiến trúc chuẩn hóa đầu vào, bảo đảm bảo đảm dữ liệu mới đưa vào đều được biến đổi đồng nhất


### 5.3 Lưu trữ Dữ liệu Phân chia & Dây chuyền Tiền xử lý
- **Xuất 2 tệp CSV trực quan:** `data/processed/train.csv` (26.128 dòng) và `data/processed/test.csv` (6.532 dòng) để người dùng và giảng viên có thể click đúp mở xem trực tiếp trên Excel/Notepad.
- **Lưu cấu trúc nhị phân:** Lưu tập phân chia (`X_train`, `X_test`, `y_train`, `y_test`) vào `data/processed/train_test_data.joblib` để máy học nạp siêu tốc.
- **Lưu dây chuyền xử lý:** Lưu `preprocessor` vào `models/saved_models/preprocessor.joblib`.
- Bàn giao đầy đủ tài nguyên đã chuẩn hóa cho Notebook 3 phục vụ công đoạn huấn luyện, đánh giá chéo phân tầng và tinh chỉnh siêu tham số.

In [ ]:
# 1. Xuất 2 tệp CSV trực quan mở được bằng Excel qua hàm export_train_test_csv()
train_csv_path, test_csv_path = export_train_test_csv(X_train, X_test, y_train, y_test)
print(f"✓ Đã xuất tập Train mở bằng Excel: {train_csv_path}")
print(f"✓ Đã xuất tập Test mở bằng Excel:  {test_csv_path}")

# 2. Lưu trữ dữ liệu nhị phân và preprocessor qua save_pipeline_and_splits()
prep_p, data_p = save_pipeline_and_splits(preprocessor, X_train, X_test, y_train, y_test)
print(f"✓ Đã lưu cấu trúc máy học: {prep_p} và {data_p}")


**Tổng kết Giai đoạn 2 :**
- Hoàn thành trích xuất 7 đặc trưng miền nghiệp vụ và chia tập Train/Test 80/20 phân tầng.
- Dây chuyền xử lý kết hợp chữ và số (2.007 đặc trưng) đã được đóng gói an toàn thành các tệp `.joblib`.
